In [1]:
using CSV, DataFrames,JSON, CodecZlib

In [2]:

#sequence,locus,stop_codon,vj_in_frame,v_frameshift,productive,rev_comp,complete_vdj,v_call,d_call,j_call,sequence_alignment,germline_alignment,sequence_alignment_aa,germline_alignment_aa,v_alignment_start,v_alignment_end,d_alignment_start,d_alignment_end,j_alignment_start,j_alignment_end,v_sequence_alignment,v_sequence_alignment_aa,v_germline_alignment,v_germline_alignment_aa,d_sequence_alignment,d_sequence_alignment_aa,d_germline_alignment,d_germline_alignment_aa,j_sequence_alignment,j_sequence_alignment_aa,j_germline_alignment,j_germline_alignment_aa,fwr1,fwr1_aa,cdr1,cdr1_aa,fwr2,fwr2_aa,cdr2,cdr2_aa,fwr3,fwr3_aa,fwr4,fwr4_aa,cdr3,cdr3_aa,junction,junction_length,junction_aa,junction_aa_length,v_score,d_score,j_score,v_cigar,d_cigar,j_cigar,v_support,d_support,j_support,v_identity,d_identity,j_identity,v_sequence_start,v_sequence_end,v_germline_start,v_germline_end,d_sequence_start,d_sequence_end,d_germline_start,d_germline_end,j_sequence_start,j_sequence_end,j_germline_start,j_germline_end,fwr1_start,fwr1_end,cdr1_start,cdr1_end,fwr2_start,fwr2_end,cdr2_start,cdr2_end,fwr3_start,fwr3_end,fwr4_start,fwr4_end,cdr3_start,cdr3_end,np1,np1_length,np2,np2_length,c_region,Redundancy,ANARCI_numbering,ANARCI_status
function process_data(path::String)
    
    df = CSV.File(path, 
              skipto=3,  # Skip the metadata line
              header=2,  # Use the second line as header
              delim=',', 
              missingstring=["", "NA"],
              ignoreemptyrows=true,
              silencewarnings=true) |> DataFrame;;
   
   return df
end

process_data (generic function with 1 method)

In [96]:
function filter_data(df::DataFrame,src::String, dst::String)
    denied = Int64[] #rows that we dont use
    filter = ["rev_comp" "productive"; true false] #2D array of headers we want to filter and what value to filter away

    println("file: ", src)
    
    for i in 1:size(filter, 2)
        header = filter[1,i]
        undesired_value = filter[2,i]
        
        if header in names(df)
            for (index, value) in enumerate(df[!, header])
                if value == undesired_value
                    push!(denied, index)
                end
            end
            println(length(denied),"/",nrow(df)," removed with condition: " ,filter[1,i], " = ", filter[2,i])
        end
    end

    if length(denied) != nrow(df)
        filtered_df = df[Not(denied), :]
        # Extract only the "sequence_alignment_aa" column
        sequence_column = filtered_df[!, "sequence_alignment_aa"]

        path= dst[1:end-6] * "txt"
        # Write only this column to the file
        open(path, "w") do io
            for sequence in sequence_column
                println(io, sequence)
            end
        end
        
    else
        println("all rows denied, not copying file over")
    end

    println("")
    
    return nrow(df)-length(denied)

#=
    if (true)# change: 
        cp(src,dst;force=true) #!!!WARNING!!! deletes destination directory and creates a new one with same name. original contents will be lost
        return true
    elseif (false)
        return false
    end
    =#
    
end

filter_data (generic function with 1 method)

In [97]:

directory = "antibody_data/unfiltered_data/"
unfiltered_data = readdir(directory)
filtered_dir = "antibody_data/filtered_data/"

accepted = 0
total_rows = 0

for i in eachindex(unfiltered_data)
    file = unfiltered_data[i]
    path = "antibody_data/unfiltered_data/$file"
    
    df = process_data(path)
   total_rows += filter_data(df, path, "$filtered_dir$file")

end
println("total amount of rows of data accepted: ", total_rows)
#print(accepted ,"/",length(unfiltered_data)," files moved into: " ,filtered_dir)

file: antibody_data/unfiltered_data/ERR220397_Heavy_Bulk.csv.gz
913/1772 removed with condition: rev_comp = true
913/1772 removed with condition: productive = false

file: antibody_data/unfiltered_data/ERR220429_Heavy_Bulk.csv.gz
35/48 removed with condition: rev_comp = true
35/48 removed with condition: productive = false

file: antibody_data/unfiltered_data/ERR220431_Heavy_Bulk.csv.gz
0/36 removed with condition: rev_comp = true
0/36 removed with condition: productive = false

total amount of rows of data accepted: 908
